<a href="https://colab.research.google.com/github/Ashton94089/Actuarial-Practice/blob/main/actuarial_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 精算預測模型：頻率-程度雙模型法 (Frequency-Severity Approach)

這個方法將總損失 ($L$) 拆解為：

$$\text{預期總損失 (Pure Premium)} = \text{預期理賠頻率 (Frequency)} \times \text{預期單次理賠金額 (Severity)}$$

*   **頻率模型 (Frequency)**：使用 **Poisson 回歸 (GLM)** 預測單位 Exposure 下的理賠次數。
*   **程度模型 (Severity)**：使用 **Gamma 回歸 (GLM)** 預測發生理賠時的平均每筆金額。

我們將模擬保險公司的原始保單資料與生命表，並以此為基礎建立上述模型。

### 1. 模擬原始保單資料與生命表

為了建構模型，我們需要模擬一份包含保戶特徵、理賠紀錄的保單資料，以及一份參考用的生命表。這將作為我們的訓練數據。

In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 設定隨機種子以確保結果可重複
np.random.seed(42)
n_policies = 1000

# 1. 建立標準生命表 (Mortality Table)
# 假設 q_x 代表該年齡的死亡/重疾發生機率
mortality_table = pd.DataFrame({
    'age': range(20, 61),
    'base_prob': [0.001 * (1.08 ** (age - 20)) for age in range(20, 61)]
})

# 2. 模擬保單原始資料 (Policyholder Data)
data = pd.DataFrame({
    'policy_id': range(1001, 1001 + n_policies),
    'age': np.random.randint(20, 60, size=n_policies),
    'gender': np.random.choice(['M', 'F'], size=n_policies, p=[0.5, 0.5]),
    'bmi': np.round(np.random.normal(24, 4, size=n_policies), 1),
    'exposure': np.random.choice([0.5, 1.0], size=n_policies, p=[0.2, 0.8]) # 保單經過年數 (Exposure)
})

# 併入生命表的基礎發生率作為特徵
data = data.merge(mortality_table, on='age', how='left')

# 模擬真實的理賠次數 (Claim Count) - 假設真實發生率受性別與 BMI 影響
lambda_param = data['base_prob'] * data['exposure'] * np.where(data['gender'] == 'M', 1.2, 1.0) * (data['bmi'] / 22)
data['claim_count'] = np.random.poisson(lambda_param)

# 模擬每次理賠的金額 (Severity) - 假設單次理賠服從 Gamma 分佈
def generate_claim_amount(count):
    if count == 0:
        return 0.0
    return np.sum(np.random.gamma(shape=2.0, scale=5000, size=count))

data['claim_amount'] = data['claim_count'].apply(generate_claim_amount)

print("--- 原始保單資料前 5 筆 ---")
display(data[['policy_id', 'age', 'gender', 'bmi', 'exposure', 'base_prob', 'claim_count', 'claim_amount']].head())

--- 原始保單資料前 5 筆 ---


,policy_id,age,gender,bmi,exposure,base_prob,claim_count,claim_amount
0,1001,58,M,19.7,1.0,0.018625,0,0.0
1,1002,48,F,16.9,1.0,0.008627,0,0.0
2,1003,34,F,29.0,0.5,0.002937,0,0.0
3,1004,27,M,24.8,1.0,0.001714,0,0.0
4,1005,40,M,23.5,0.5,0.004661,0,0.0


### 2. 理賠頻率模型 (Poisson GLM)

我們將使用 Poisson 廣義線性模型來預測保單的理賠次數。Poisson 分佈常用於模擬計數數據（例如事件發生的次數）。在這裡，`log(exposure)` 被用作 `offset`，這意味著模型預測的是單位風險暴露下的理賠率。

**模型公式**：`claim_count ~ C(gender) + bmi + np.log(base_prob)`

*   `claim_count`：因變量，即理賠次數。
*   `C(gender)`：性別作為類別變量。
*   `bmi`：身體質量指數作為連續變量。
*   `np.log(base_prob)`：生命表的基礎發生率的對數，反映潛在風險。
*   `exposure`：風險暴露量，例如保單的有效年數。我們會將其納入模型的 `offset` 參數，以便預測年化理賠率。

In [5]:
# ----------------------------------------------------
# 步驟 A: 理賠頻率模型 (Poisson GLM)
# ----------------------------------------------------
# 使用 log(exposure) 作為 offset，計算年化理賠率
freq_model = smf.glm(
    formula="claim_count ~ C(gender) + bmi + np.log(base_prob)",
    data=data,
    family=sm.families.Poisson(),
    exposure=data['exposure']
).fit()

print("--- 頻率模型摘要（局部） ---")
print(freq_model.summary().tables[1])

--- 頻率模型摘要（局部） ---
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -2.3628      3.484     -0.678      0.498      -9.191       4.465
C(gender)[T.M]        1.1561      0.817      1.414      0.157      -0.446       2.758
bmi                   0.1194      0.082      1.452      0.147      -0.042       0.281
np.log(base_prob)     1.2397      0.554      2.236      0.025       0.153       2.326


### 3. 理賠金額程度模型 (Gamma GLM)

接下來，我們需要預測每次理賠的平均金額。由於理賠金額通常是右偏的連續數據，Gamma 分佈的廣義線性模型是合適的選擇。我們只對**有發生理賠**的保單進行程度模型的訓練。

**模型公式**：`avg_claim_severity ~ age + C(gender) + bmi`

*   `avg_claim_severity`：因變量，即單次理賠的平均金額。
*   `age`：年齡作為連續變量。
*   `C(gender)`：性別作為類別變量。
*   `bmi`：身體質量指數作為連續變量。

我們使用 `Log` 連結函數，這是 Gamma 分佈常用的連結函數。

In [6]:
# ----------------------------------------------------
# 步驟 B: 理賠金額程度模型 (Gamma GLM)
# ----------------------------------------------------
# 僅對「有發生理賠」的保單訓練程度模型
claims_data = data[data['claim_count'] > 0].copy()
claims_data['avg_claim_severity'] = claims_data['claim_amount'] / claims_data['claim_count']

sev_model = smf.glm(
    formula="avg_claim_severity ~ age + C(gender) + bmi",
    data=claims_data,
    family=sm.families.Gamma(link=sm.families.links.Log())
).fit()

print("--- 程度模型摘要（局部） ---")
print(sev_model.summary().tables[1])

--- 程度模型摘要（局部） ---
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          7.9678      1.776      4.487      0.000       4.487      11.449
C(gender)[T.M]     0.9712      0.388      2.501      0.012       0.210       1.732
age               -0.0131      0.025     -0.528      0.598      -0.062       0.036
bmi                0.0340      0.044      0.780      0.436      -0.052       0.120


### 4. 預測新保戶/未來的期望損失 (Expected Pure Premium)

模型訓練完成後，我們就可以使用這兩個模型來評估一份「新保單」在未來一段時間（例如一年）的預期損失金額。這個預期損失金額是保費定價的重要基礎。

In [7]:
# 新保戶資料 (未來一年的新保單，exposure = 1.0)
new_policies = pd.DataFrame({
    'age': [25, 55],
    'gender': ['F', 'M'],
    'bmi': [21.5, 28.0],
    'exposure': [1.0, 1.0] # 假設預測未來一年的風險暴露
})

# 併入生命表基礎機率 (需要確保 age 欄位匹配)
# 注意：這裡假設 new_policies 的 age 會在 mortality_table 的範圍內
new_policies = new_policies.merge(mortality_table, on='age', how='left')

# 1. 預測未來一年的理賠次數 (頻率)
new_policies['pred_frequency'] = freq_model.predict(new_policies)

# 2. 預測每次理賠的平均金額 (程度)
new_policies['pred_severity'] = sev_model.predict(new_policies)

# 3. 計算預期純保費 / 未來純損失 (Pure Premium)
new_policies['expected_loss'] = new_policies['pred_frequency'] * new_policies['pred_severity']

print("\n--- 新保戶未來一年損失預測結果 ---")
display(new_policies[['age', 'gender', 'bmi', 'pred_frequency', 'pred_severity', 'expected_loss']])


--- 新保戶未來一年損失預測結果 ---


,age,gender,bmi,pred_frequency,pred_severity,expected_loss
0,25,F,21.5,0.000378,4325.798995,1.634080
1,55,M,28.0,0.045656,9622.630158,439.329957


這個範例展示了精算師/資料科學家在風險定價中的核心工作：

1.  **資料模擬與處理**：從無到有建立符合實務情境的數據。
2.  **模型選擇**：根據數據特性選擇合適的統計模型 (Poisson for count, Gamma for continuous positive skew).
3.  **參數解釋**：理解模型係數的意義。
4.  **結果應用**：將模型預測結果轉換為業務決策（例如計算 `expected_loss` 作為保費定價基礎）。

您可以將此專案擴展，例如嘗試不同的特徵、不同的模型（如機器學習模型），或進行更詳細的結果分析和視覺化。祝您面試順利！